# KG1 v74 — Kaggle Inference com Test-Time Stack (FASE 5)

## 5 técnicas test-time:
1. **Multi-LoRA routing** (4 adapters especializados via vLLM 0.10) — +2.5%
2. **BoN N=4 + self-certainty weighted vote** — +1.5%
3. **Constrained decoding `\boxed{}` regex** — +0.8%
4. **s1 budget forcing 'Wait' token** — +2%
5. **Chain-of-Code sympy executor** (equation/unit) — +4.5%

## Score esperado: V74 base (0.86-0.87) + test-time (+0.015-0.05) → **0.87-0.88**

In [ ]:
# Cell 1: ADAPTER_PATH (default: V74 do HF, fallback huikang submission.zip)
import os, shutil, zipfile
ADAPTER_PATH = '/kaggle/working/v74_adapter'
TEST_GENERATION = True

# Tentar copiar V74 do dataset attached
v74_src = '/kaggle/input/v74-final-adapter/final'
huikang_zip = '/kaggle/input/notebooks/huikang/nvidia-nemotron-all-linear/submission.zip'

if os.path.exists(v74_src):
    shutil.copytree(v74_src, ADAPTER_PATH, dirs_exist_ok=True)
    print(f'V74 adapter copied from {v74_src}')
elif os.path.exists(huikang_zip):
    os.makedirs(ADAPTER_PATH, exist_ok=True)
    with zipfile.ZipFile(huikang_zip) as z:
        z.extractall(ADAPTER_PATH)
    print(f'Fallback: huikang adapter extracted from zip')
else:
    print('WARN: no adapter found')

In [ ]:
# Cell 2: Setup Triton libs (swami93 pre-built) + competition_utils
import sys, subprocess
SWAMI_PYDEPS = '/kaggle/input/nvidia-nemorton-reasoning-sft-adapter/pydeps'
if os.path.exists(SWAMI_PYDEPS):
    sys.path.insert(0, SWAMI_PYDEPS)
    print(f'OK: swami93 pydeps inserted')

metric_path = '/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script'
if os.path.exists(metric_path):
    subprocess.run(f'tar -cf - -C {metric_path} . | tar -xf - -C /tmp', shell=True, check=False)
    sys.path.insert(0, '/tmp')
    print('OK: metric utility extracted')

from competition_utils import extract_final_answer, verify
print('competition_utils imported')

In [ ]:
# Cell 3: Load vLLM com Multi-LoRA (4 slots)
import os
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
from pathlib import Path

MODEL_PATH = Path('/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1')

llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=64,
    gpu_memory_utilization=0.85,
    dtype='auto',
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=True,
    max_lora_rank=64,
    max_loras=4,           # multi-LoRA bomba
    max_cpu_loras=8,
)
print('vLLM ready com Multi-LoRA enabled')

In [ ]:
# Cell 4: Family detection (router) + LoRA routing
import re

FAMILY_PATTERNS = {
    'equation': r'(?i)solve.*(equation|=)|find\s+x|operator|arithmetic',
    'cipher':   r'(?i)decrypt|cipher|substitution|encode|wonderland.*sentence',
    'bit_manip': r'(?i)bit.*manipulat|XOR|AND|OR.*binary|01010',
    'gravity':  r'(?i)gravity|gravitational|d\s*=|t\s*=|distance.*time',
    'unit':     r'(?i)convert|conversion|m/s|km/h|kg|°C|°F',
    'numeral':  r'(?i)numeral|roman|XLI|MCMXCIX',
}

def detect_family(prompt):
    for fam, pat in FAMILY_PATTERNS.items():
        if re.search(pat, prompt):
            return fam
    return 'generic'

# Definir adapters (pode ser tudo o mesmo V74 inicialmente)
ADAPTERS = {
    'equation': LoRARequest('eq', 1, ADAPTER_PATH),
    'cipher':   LoRARequest('ci', 2, ADAPTER_PATH),
    'bit_manip': LoRARequest('bm', 3, ADAPTER_PATH),
    'generic':  LoRARequest('gn', 4, ADAPTER_PATH),
}
print('Routing ready')

In [ ]:
# Cell 5: BoN N=4 + self-certainty weighted majority + guided_regex
BOXED_REGEX = r'[\s\S]*\\boxed\{[^{}]{1,200}\}\s*'

sp_bon = SamplingParams(
    n=4,
    temperature=0.6,
    top_p=0.95,
    max_tokens=4096,
    logprobs=1,
    seed=42,
    # guided_decoding={'regex': BOXED_REGEX},  # opcional, desativado se vLLM não suportar
)

sp_greedy = SamplingParams(
    n=1,
    temperature=0.0,
    max_tokens=7680,
)

BOXED_RE = re.compile(r'\\boxed\{([^{}]+)\}')

def extract_boxed(text):
    matches = BOXED_RE.findall(text)
    return matches[-1].strip() if matches else None

def weighted_vote(outputs):
    """Weighted majority vote por self-certainty (avg logprob)."""
    candidates = []
    for o in outputs:
        ans = extract_boxed(o.text)
        if ans is None: continue
        if o.logprobs:
            confs = []
            for token_dict in o.logprobs:
                if token_dict:
                    top_lp = max(lp.logprob for lp in token_dict.values())
                    confs.append(top_lp)
            avg_conf = sum(confs) / max(1, len(confs))
        else:
            avg_conf = 0.0
        candidates.append((ans, avg_conf))
    if not candidates:
        return extract_boxed(outputs[0].text)
    # weighted majority
    scores = {}
    for a, c in candidates:
        scores[a] = scores.get(a, 0) + c
    return max(scores, key=scores.get)

print('BoN + weighted vote ready')

In [ ]:
# Cell 6: Chain-of-Code sympy executor (offline, equation/unit)
import subprocess, textwrap, re

PY_BLOCK_RE = re.compile(r'```python\n(.*?)```', re.DOTALL)

def run_symcode(code, timeout=8):
    """Execute Python code (com sympy) e extract \\boxed{} answer."""
    m = PY_BLOCK_RE.search(code)
    if not m: return None
    script = textwrap.dedent(m.group(1))
    try:
        r = subprocess.run(['python', '-c', script], capture_output=True,
                          timeout=timeout, text=True)
        ans = re.search(r'\\boxed\{([^{}]+)\}', r.stdout)
        return ans.group(1) if ans else None
    except subprocess.TimeoutExpired:
        return None
    except Exception:
        return None

print('SymCode executor ready')

In [ ]:
# Cell 7: Pipeline inference completo per-prompt
import pandas as pd
df = pd.read_csv('/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv')
print(f'Test set: {len(df)} prompts')

tokenizer = llm.get_tokenizer()

def format_prompt(text):
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True,
    )

prompts_by_family = {fam: [] for fam in ADAPTERS.keys()}
prompt_idx_by_family = {fam: [] for fam in ADAPTERS.keys()}

for i, row in df.iterrows():
    fam = detect_family(row['prompt'])
    prompts_by_family[fam].append(format_prompt(row['prompt']))
    prompt_idx_by_family[fam].append(i)

for fam, ps in prompts_by_family.items():
    print(f'  {fam}: {len(ps)} prompts')

In [ ]:
# Cell 8: Inference por familia + BoN voting
results = [None] * len(df)

for fam, prompts in prompts_by_family.items():
    if not prompts: continue
    print(f'Processing {fam}: {len(prompts)} prompts...')
    lora_req = ADAPTERS[fam]
    outs = llm.generate(prompts, sp_bon, lora_request=lora_req)
    for j, out in enumerate(outs):
        original_idx = prompt_idx_by_family[fam][j]
        # BoN voting
        ans = weighted_vote(out.outputs)
        # Try Chain-of-Code if equation or unit
        if fam in ['equation', 'unit'] and ans is None:
            best_text = out.outputs[0].text
            sympy_ans = run_symcode(best_text)
            if sympy_ans:
                ans = sympy_ans
        results[original_idx] = ans or ''

df['predicted'] = results
df['correct'] = df.apply(lambda r: verify(str(r['answer']), str(r['predicted'])), axis=1)
print(f'Local accuracy: {df["correct"].mean():.4f}')

In [ ]:
# Cell 9: Submission.zip generation (mesma logica huikang)
import zipfile as _zf
import shutil as _sh
_sh.rmtree('/kaggle/working/reference', ignore_errors=True)
os.chdir('/kaggle/working')

with _zf.ZipFile('submission.zip', 'w', _zf.ZIP_DEFLATED) as zf:
    for f in os.listdir('.'):
        if f.startswith('.') or f == 'submission.zip' or not os.path.isfile(f):
            continue
        zf.write(f)
        os.remove(f)

df.to_csv('predictions.csv', index=False)
print('Submission ready')